In [1]:
import pandas as pd
import os
import subprocess
from Bio.Blast.Applications import NcbimakeblastdbCommandline, NcbiblastpCommandline, NcbiblastnCommandline,NcbiblastxCommandline

In [2]:
species='13894 94328 3656 3888 3847 4081'.split()
exosome_dic={'13894':"SRR7406450",
            "94328":"SRR7406451",
            "3656":"SRR7406453",
            "3888":"SRR7406455",
            "3847":"SRR7406458",
            "4081":"SRR7406459",
           }

In [3]:
makeblastdb_path = "C:\\Program Files\\NCBI\\blast-2.16.0+\\bin\\makeblastdb.exe"
db_path='C:\\Users\\huangyan8\\Desktop\\work\\2025-04-04_HT_TE_Seqs\\db\\'
# 定义文件路径
blastn_path="C:\\Program Files\\NCBI\\blast-2.16.0+\\bin\\blastn.exe"
seq_save_path='E:\\Linux_Work_Project\\TE_HT\\10.Family_Specie_Sample_HT_Seqs\\02.Specie_Seqs\\'

In [ ]:
for specie in species:
    query_fasta_file ='E:\\Linux_Work_Project\\TE_HT\\09.Exosome\\'+exosome_dic[specie]+'.fa'
    print(specie)
    for sample in os.listdir(seq_save_path+specie):
        print(sample)
        files=os.listdir(seq_save_path+specie+"\\"+sample)
        for file in files:
            print(file)
            if specie+"_"+sample+"_"+file.split(".fa")[0]+'_db'+".ndb" not in os.listdir(db_path):
                fasta_file=seq_save_path+specie+"\\"+sample+"\\"+file
                makeblastdb_cmd = NcbimakeblastdbCommandline(
                    cmd=makeblastdb_path,
                    dbtype="nucl",
                    input_file=fasta_file,
                    out=db_path+specie+"_"+sample+"_"+file.split(".fa")[0]+'_db'
                )

                try:
                    # 运行命令并捕获输出
                    process = subprocess.Popen(
                        str(makeblastdb_cmd),
                        stdout=subprocess.PIPE,
                        stderr=subprocess.PIPE,
                        shell=True,
                        text=True,  # 设置为文本模式
                        encoding='utf-8'  # 指定编码为utf-8
                    )
                    stdout, stderr = process.communicate()

                    # 打印输出
                    print("标准输出:\n", stdout)
                    print("标准错误输出:\n", stderr)

                    # 检查返回码
                    if process.returncode != 0:
                        print(f"makeblastdb 命令返回非零退出代码: {process.returncode}")
                        print("标准错误输出:\n", stderr)

                except subprocess.CalledProcessError as e:
                    print("子进程调用错误:", e)
                    print("返回代码:", e.returncode)
                    print("标准输出:\n", e.stdout)
                    print("标准错误输出:\n", e.stderr)
                except Exception as e:
                    print("其他错误:", e)

                print("BLAST数据库创建完成")

            # Blastn
            database_fasta_file =db_path+specie+"_"+sample+"_"+file.split(".fa")[0]+'_db'
            out_file=specie+"_"+sample+"_"+file.replace(".fa",'')+"_blastn_results.csv"
            blast_output_file = 'C:\\Users\\huangyan8\\Desktop\\work\\2025-04-04_HT_TE_Seqs\\Result\\'+out_file
            # 自定义输出格式字符串
            if out_file not in os.listdir("./Result"):
                custom_outfmt = '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore'
                # 运行BLAST比对
                blastn_cmd = NcbiblastnCommandline(
                        cmd=blastn_path,
                        query=query_fasta_file,
                        db=database_fasta_file,
                        outfmt=custom_outfmt,
                        evalue=1e-5,
                        max_target_seqs=5,
                        out=blast_output_file,
                        num_threads=8 # 使用4个线程
                    )
                blastn_cmd()
                #print("BLAST比对完成，结果已保存到:\t"+file.replace(".fa",'')+"_blastn_results.csv")
                r=open(blast_output_file,'r').readlines()
                print(file,"Blast Result Count:\t",len(r))